# 01 — Verify ACIL analytic update

Kiểm chứng số học trên dữ liệu tổng hợp, không phải tái lập benchmark ảnh của paper.
Joint được cài trước và giải ridge trực tiếp; Incremental chỉ nhận trạng thái R, W cùng phase hiện tại.
Cùng feature map cố định, gamma và float64. Ba phase, mỗi phase thêm hai lớp.
Nguồn công thức: Zhuang et al., ACIL, NeurIPS 2022, mục 3.1–3.2, phương trình (4), (9), (10).


In [1]:
import numpy as np
import sys

SEED = 42
GAMMA = 1.0
D_INPUT = 16
D_FEATURE = 24
N_PHASES = 3
CLASSES_PER_PHASE = 2
N_TRAIN = 20  # mỗi lớp
N_TEST = 10   # mỗi lớp, độc lập với train
rng = np.random.default_rng(SEED)
W_fe = rng.normal(size=(D_INPUT, D_FEATURE)).astype(np.float64)

# Cùng feature map cho train, test, joint và incremental.
def feature_map(x):
    """Biến x (n,16) thành đặc trưng ReLU (n,24) bằng W_fe cố định."""
    return np.maximum(x @ W_fe, 0.0)

# Dữ liệu các phase được giữ bởi bộ kiểm thử, không bởi hàm incremental.
phases = []
for k in range(N_PHASES):
    train_x, train_y, test_x, test_y = [], [], [], []
    for local_label in range(CLASSES_PER_PHASE):
        label = k * CLASSES_PER_PHASE + local_label
        center = rng.normal(0, 2, size=D_INPUT)
        train_x.append(center + rng.normal(size=(N_TRAIN, D_INPUT)))
        test_x.append(center + rng.normal(size=(N_TEST, D_INPUT)))
        train_y.extend([label] * N_TRAIN)
        test_y.extend([label] * N_TEST)
    phases.append((feature_map(np.vstack(train_x)), np.array(train_y),
                   feature_map(np.vstack(test_x)), np.array(test_y)))
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__)
print('Feature dimension:', D_FEATURE, '| dtype:', phases[0][0].dtype)


Python: 3.13.15 | NumPy: 2.1.3
Feature dimension: 24 | dtype: float64


## Nhánh Joint — kết quả tham chiếu
Gộp tất cả dữ liệu đã thấy. Không sử dụng R hoặc W của nhánh incremental.

In [2]:
def joint_ridge(h, labels, n_classes, gamma):
    """Giải ridge trực tiếp từ h (n,d), nhãn toàn cục; trả W (d,C)."""
    y = np.eye(n_classes, dtype=np.float64)[labels]
    a = h.T @ h + gamma * np.eye(h.shape[1], dtype=np.float64)
    return np.linalg.solve(a, h.T @ y)

joint_weights = []
for k in range(N_PHASES):
    h_seen = np.vstack([phase[0] for phase in phases[:k + 1]])
    y_seen = np.concatenate([phase[1] for phase in phases[:k + 1]])
    joint_weights.append(joint_ridge(
        h_seen, y_seen, (k + 1) * CLASSES_PER_PHASE, GAMMA))
print('Joint reference ready:', [w.shape for w in joint_weights])


Joint reference ready: [(24, 2), (24, 4), (24, 6)]


## Nhánh Incremental
Nhãn đầu vào one-hot chỉ gồm các lớp mới. Hai khối trọng số được ghép ngang; R phải được cập nhật trước W.

In [3]:
def initialize(h, y_local, gamma):
    """Khởi tạo từ phase 0: h (n,d), one-hot y_local (n,c); trả R,W."""
    a = h.T @ h + gamma * np.eye(h.shape[1], dtype=np.float64)
    r = np.linalg.solve(a, np.eye(h.shape[1], dtype=np.float64))
    return r, r @ h.T @ y_local


def incremental_update(r_old, w_old, h, y_local):
    """Nhận R,W cũ và dữ liệu phase mới; trả R,W mới theo ACIL (9),(10)."""
    b = r_old @ h.T
    middle = np.eye(h.shape[0], dtype=np.float64) + h @ b
    r_new = r_old - b @ np.linalg.solve(middle, h @ r_old)
    w_old_updated = w_old - r_new @ h.T @ (h @ w_old)
    w_new_classes = r_new @ h.T @ y_local
    return r_new, np.hstack((w_old_updated, w_new_classes))


## Đối chiếu sau mỗi phase
Kiểm tra W, điểm dự đoán, nhãn trên test độc lập và R so với định nghĩa trực tiếp. Accuracy chỉ mô tả dữ liệu tổng hợp, không phải điểm ACIL trên dữ liệu ảnh.

In [4]:
results = []
r = w = None
for k, (h, labels, _, _) in enumerate(phases):
    local_labels = labels - k * CLASSES_PER_PHASE
    y_local = np.eye(CLASSES_PER_PHASE, dtype=np.float64)[local_labels]
    if k == 0:
        r, w = initialize(h, y_local, GAMMA)
    else:
        r, w = incremental_update(r, w, h, y_local)
    n_classes = (k + 1) * CLASSES_PER_PHASE
    assert w.shape == (D_FEATURE, n_classes)
    assert r.shape == (D_FEATURE, D_FEATURE)
    assert r.dtype == w.dtype == np.float64
    w_joint = joint_weights[k]
    h_test = np.vstack([p[2] for p in phases[:k + 1]])
    y_test = np.concatenate([p[3] for p in phases[:k + 1]])
    s_joint, s_inc = h_test @ w_joint, h_test @ w
    h_seen = np.vstack([p[0] for p in phases[:k + 1]])
    r_reference = np.linalg.solve(
        h_seen.T @ h_seen + GAMMA * np.eye(D_FEATURE), np.eye(D_FEATURE))
    np.testing.assert_allclose(w, w_joint, rtol=1e-8, atol=1e-10)
    np.testing.assert_allclose(r, r_reference, rtol=1e-8, atol=1e-10)
    np.testing.assert_allclose(s_inc, s_joint, rtol=1e-8, atol=1e-10)
    np.testing.assert_array_equal(s_inc.argmax(1), s_joint.argmax(1))
    row = dict(phase=k, n_classes=n_classes,
               max_abs_w_error=float(np.max(np.abs(w - w_joint))),
               relative_w_error=float(np.linalg.norm(w - w_joint) / np.linalg.norm(w_joint)),
               max_abs_score_error=float(np.max(np.abs(s_inc - s_joint))),
               test_accuracy=float(np.mean(s_inc.argmax(1) == y_test)))
    results.append(row)
    print(row)
print('PASS: joint và incremental tương đương trong ngưỡng kiểm tra.')


{'phase': 0, 'n_classes': 2, 'max_abs_w_error': 1.8395007739258062e-14, 'relative_w_error': 2.1553909202805462e-13, 'max_abs_score_error': 7.682743330406083e-14, 'test_accuracy': 1.0}
{'phase': 1, 'n_classes': 4, 'max_abs_w_error': 1.3648804308985518e-14, 'relative_w_error': 1.9482928534428143e-13, 'max_abs_score_error': 1.5432100042289676e-13, 'test_accuracy': 1.0}
{'phase': 2, 'n_classes': 6, 'max_abs_w_error': 1.3239409568654992e-14, 'relative_w_error': 1.2571586699457025e-13, 'max_abs_score_error': 1.8096635301390052e-13, 'test_accuracy': 1.0}
PASS: joint và incremental tương đương trong ngưỡng kiểm tra.


In [5]:
import platform
import json
from pathlib import Path

report = {
    "experiment": "VERIFY-01",
    "description": "Joint versus incremental ridge on synthetic data",
    "config": {
        "seed": SEED,
        "gamma": GAMMA,
        "d_input": D_INPUT,
        "d_feature": D_FEATURE,
        "n_phases": N_PHASES,
        "classes_per_phase": CLASSES_PER_PHASE,
        "n_train_per_class": N_TRAIN,
        "n_test_per_class": N_TEST,
        "dtype": "float64",
    },
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
    },
    "results": results,
    "verification_passed": True,
    "wandb_status": "not_synced",
}

output_path = Path("/kaggle/working/verify_analytic_update_results.json")
output_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Đã lưu:", output_path)

Đã lưu: /kaggle/working/verify_analytic_update_results.json


## Theo dõi W&B khi chạy trên Kaggle

Chạy cell dưới sau khi thêm secret `WANDB_API_KEY` trong Kaggle và bật Internet.
Điền entity và project công khai của nhóm. Bật LOG_WANDB để gửi một run mới.
Không đưa API key vào notebook. Cell mặc định không gửi dữ liệu từ môi trường tạo file.
Sau khi chạy, ghi URL run vào bảng thí nghiệm trong README cùng commit GitHub tương ứng.


In [6]:
%pip install -q wandb

Note: you may need to restart the kernel to use updated packages.


In [7]:
LOG_WANDB = True
WANDB_ENTITY = "hsangg-2006"
WANDB_PROJECT = "acil-reproduction"

if LOG_WANDB:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
    with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                    job_type='verification', tags=['verification', 'synthetic', 'float64'],
                    config=dict(seed=SEED, gamma=GAMMA, d_input=D_INPUT,
                                d_feature=D_FEATURE, n_phases=N_PHASES,
                                classes_per_phase=CLASSES_PER_PHASE,
                                numpy_version=np.__version__, dtype='float64')) as run:
        for row in results:
            run.log(row, step=row['phase'])
        print('W&B run:', run.url)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hsangg2006 (hsangg-2006) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run eyg1alwi
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261102_053947-eyg1alwi
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dark-music-3
wandb: ⭐️ View project at https://wandb.ai/hsangg-2006/acil-reproduction
wandb: 🚀 View run at https://wandb.ai/hsangg-2006/acil-reproduction/runs/eyg1alwi
wandb: updating run metadata; uploading summary


W&B run: https://wandb.ai/hsangg-2006/acil-reproduction/runs/eyg1alwi


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt
wandb: 
wandb: Run history:
wandb: max_abs_score_error ▁▆█
wandb:     max_abs_w_error █▂▁
wandb:           n_classes ▁▅█
wandb:               phase ▁▅█
wandb:    relative_w_error █▆▁
wandb:       test_accuracy ▁▁▁
wandb: 
wandb: Run summary:
wandb: max_abs_score_error 0.0
wandb:     max_abs_w_error 0.0
wandb:           n_classes 6
wandb:               phase 2
wandb:    relative_w_error 0.0
wandb:       test_accuracy 1
wandb: 
wandb: 🚀 View run dark-music-3 at: https://wandb.ai/hsangg-2006/acil-reproduction/runs/eyg1alwi
wandb: ⭐️ View project at: https://wandb.ai/hsangg-2006/acil-reproduction
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261102_053947-eyg1alwi/logs


In [8]:
import sys
import platform
import numpy as np

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Platform:", platform.platform())
print("Seed:", SEED)
print("Gamma:", GAMMA)
print("Feature dimension:", D_FEATURE)
print("Dtype:", phases[0][0].dtype)

Python: 3.13.15
NumPy: 2.1.3
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
Seed: 42
Gamma: 1.0
Feature dimension: 24
Dtype: float64


## Kết luận

Nếu tất cả assert đạt: các công thức cập nhật đã được kiểm chứng trên cấu hình nhỏ này.
Đây không phải chứng minh lý thuyết tổng quát hay tái lập kết quả benchmark của paper.
Dữ liệu cũ chỉ được bộ kiểm thử giữ để xây dựng nhánh Joint; incremental_update không nhận dữ liệu cũ.
Bước tiếp theo: thay dữ liệu tổng hợp bằng đặc trưng CNN cố định và tái lập các thí nghiệm đã chọn.
